In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.sparse import csr_matrix

In [2]:
df = pd.read_csv('random_state158.csv')
df.head()

,userId_x,movieId,rating,timestamp_x,userId_y,tag,timestamp_y,title,genres,imdbId,tmdbId
0,17682,31433,3.5,1544094795,8360,hot guys,1289426369,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
1,17682,31433,3.5,1544094795,23053,family,1602746509,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
2,17682,31433,3.5,1544094795,23053,romance,1602746506,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
3,17682,31433,3.5,1544094795,60001,watched 2007,1187719211,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
4,17682,31433,3.5,1544094795,70971,Debra Messing,1286208696,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0


#### Std Deviation Problem

Users with st dev = 0

#### Situation 1

User with 1 rating, mean = rating, ex : rating = (3), mean = 3/1=3,  stdev = (rating-mean)/n = (3-3)/1=0

#### Situation 2

User with same rating acrross all ratings, ex: ratings = (2,2,2), mean = 6/3=2,  stdev =(rating-mean)/n = (2-2)/3=0

In [5]:
# this line creates std dev  just by n, in this sense (n - 0 = n) as in n-ddof degrees of freedom
# formula for std would be (x-xmean)^2/ N samples -degrees of freedom, degrees of freedom = 0, with ddof=0

user_stats = (df.groupby('userId_x').agg(user_mean=('rating', 'mean'),user_std=('rating', lambda x: x.std(ddof=0))).reset_index())


# this line uses std dev with denominator (n-1), this creates more nans for single raters as in n-1, becomes 1-1 so denominator becomes 0
    #user_stats = df.groupby('userId_x')['rating'].agg(['mean', 'std']).reset_index()
    #user_stats.rename(columns={'mean': 'user_mean', 'std': 'user_std'}, inplace=True)

# users with std dev = 0
display(user_stats.head())
num_zero_std = (user_stats['user_std'] == 0).sum()
print(f'Users with std dev = 0 are:{num_zero_std} out of: {len(user_stats)} or ({num_zero_std/len(user_stats):.2%})')
total_users = len(user_stats)

# Single raters,  which are a subset of the users with std dev = 0
ratings_per_user = df.groupby('userId_x')['movieId'].nunique().reset_index(name='num_ratings')
display(ratings_per_user.head())
single_raters = (ratings_per_user['num_ratings'] == 1).sum()
#print(single_raters)
print(f'Users with only one rating: {single_raters} ({single_raters/len(ratings_per_user):.2%})')

,userId_x,user_mean,user_std
0,7,3.0,0.0
1,9,4.0,0.0
2,10,3.0,0.0
3,12,2.0,0.0
4,16,2.0,0.0


Users with std dev = 0 are:48711 out of: 75598 or (64.43%)


,userId_x,num_ratings
0,7,1
1,9,1
2,10,2
3,12,1
4,16,1


Users with only one rating: 44259 (58.55%)


### Real Ratings Method, no standardization, because of the std dev = 0, problem

Formula: user mean + (movie mean - global mean)


came from : rating for movie i by user u =global mean ratings +user bias + movie bias

user bias = user mean - global mean

movie bias = movie mean - global mean


In [7]:
df_copy = df.copy(deep=True)

global_mean = df_copy['rating'].mean()

# creating series of user mean and movie mean
user_mean = df_copy.groupby('userId_x')['rating'].mean().rename('user_mean')
movie_mean = df_copy.groupby('movieId')['rating'].mean().rename('movie_mean')


#Merging series user mean and movie mean to the df_copy dataframe
df_copy = df_copy.merge(user_mean, on='userId_x').merge(movie_mean, on='movieId')
df_copy = df_copy.drop_duplicates(subset=['userId_x', 'movieId'])

#predicting the movie rating by  user_mean + (rating movie mean-global_mean) global mean = mean of all ratings
df_copy['pred_rating'] = df_copy['user_mean'] + (df_copy['movie_mean'] - global_mean)


df_copy['user_bias'] = df_copy['user_mean'] - global_mean
df_copy['movie_bias'] = df_copy['movie_mean'] - global_mean




,userId_x,movieId,rating,timestamp_x,userId_y,tag,timestamp_y,title,genres,imdbId,tmdbId,user_mean,movie_mean,pred_rating,user_bias,movie_bias
0,17682,31433,3.5,1544094795,8360,hot guys,1289426369,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0,0.677197,4.250000,1.065591,-3.184409,0.388395
27,192558,1754,5.0,1105656846,6296,Fallen Angel,1295747877,Fallen (1998),Crime|Drama|Fantasy|Thriller,119099,9411.0,4.094012,3.532258,3.764665,0.232407,-0.329347
132,26576,5013,3.0,1377399565,1551,ensemble cast,1618090996,Gosford Park (2001),Comedy|Drama|Mystery,280707,5279.0,3.104545,3.700000,2.942940,-0.757060,-0.161605
272,35895,2723,4.0,1115821789,11089,parody,1443790132,Mystery Men (1999),Action|Comedy|Fantasy,132347,9824.0,4.593525,3.093023,3.824943,0.731920,-0.768582
399,63586,5995,0.5,1447705624,1149,Roman Polanski,1424117703,"Pianist, The (2002)",Drama|War,253474,423.0,4.096680,4.128099,4.363174,0.235074,0.266494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103529358,7089,5956,4.0,1519325934,2293,Victorian era,1318294669,Gangs of New York (2002),Crime|Drama,217505,3131.0,4.000000,3.432203,3.570598,0.138395,-0.429402
103529630,152252,428,3.0,847272310,5292,Eric's Dvds,1145843095,"Bronx Tale, A (1993)",Drama,106489,1607.0,3.945534,3.568182,3.652110,0.083929,-0.293423
103529780,54603,223,2.0,891473203,132,crude humor,1284917286,Clerks (1994),Comedy,109445,2292.0,1.762332,3.809524,1.710250,-2.099273,-0.052081
103530490,83718,485,4.0,845605698,3231,action,1654197958,Last Action Hero (1993),Action|Adventure|Comedy|Fantasy,107362,9593.0,4.000000,2.750000,2.888395,0.138395,-1.111605


5

In [8]:
#display(df_copy, 5)
#df_copy.shape # (149177, 16)


(149177, 16)

### Matrix Factorization, blue or red?

User x Ratings = user latent features x movie latent features

**UxR**(userxratings) = **P** (user x k) x **Q**(movie x k)


In [10]:
#user-rating matrix
UxR = df_copy.pivot(index='userId_x', columns='movieId', values='rating').fillna(0)
UxR_sparse = csr_matrix(UxR.values)

In [11]:
UxR.shape # (75598, 12301)

(75598, 12301)

In [12]:
#display(UxR, 5)

movieId,1,2,3,4,5,6,7,8,9,10,...,286905,287665,287699,287731,287823,288265,288311,288513,288679,290573
userId_x,,,,,,,,,,,,,,,,,,,,,
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200935,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
200941,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
200943,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


5

#### Pre Gradient Descent Initialization, creating random P and Q

In [ ]:
#Users = rows, movies = columns UR = userxmovies, m x n 
num_users, num_movies = UxR.shape

k = 20  # number of latent factors/features


# Random initialization of users and movies latent features

# P =, num user x number of factors (k=20) to be adjusted by gradient descent

# Basically we are just creating a random starting point for gradient descent to start working and find the optimal values

# creating a random normal P matrix, mean centered at 0, with std dev of 0.1 within those values, of size users x k

P = np.random.normal(0, 0.1, (num_users, k))  # user latent matrix


# Q = movie latent features, movies x latent factors (k=20), to be adjusted by gradient descent
# Creating a random normal Q matrix
Q = np.random.normal(0, 0.1, (num_movies, k))  # item latent matrix

### Gradient Descent

Error term  of user u on movie i = true value of user u on movie i (-) the dot product of the latent feature vector of user u  and latent feature vector of movie  i.

Loss function:

$$
L = \sum_{(u,i) \in \mathcal{K}} 
\left( r_{u,i} - \hat{r}_{u,i} \right)^2 
+ \lambda \left( \| P \|^2 + \| Q \|^2 \right)
$$

### OLD stuff that might be usefull but with some tweaks

Method for the recommender from before, with only :

**(user mean  + movie mean - global mean)** 

or 

**global mean +(user mean-global) mean + (movie mean-global mean)** 

the first is the simplified vesion, they are both the same formula

#------OLD STUFF-----------#  DONT RUN, YET


def recommend_unrated_top_n(user_id, n=5):
    
    # look for the user_id on df copy, grab the user_id mean column, grab the first value
    user_mean = df_copy.loc[df_copy['userId_x']== user_id, 'user_mean'].iloc[0]
    
    
    # all movies df, we need user specific ratings for every movie
    all_movies = df_copy[['movieId', 'title', 'movie_mean']].drop_duplicates()

    
    # User specific prediction for all movies
    all_movies[str(user_id)+'_pred_rating'] = user_mean + (all_movies['movie_mean'] - global_mean)
   
    # Remove movies this user already rated, naming options: (seen-unseen, watched-unwatched, rated-unrated, old-new)
    rated_movies = df_copy[df_copy['userId_x'] == user_id]['movieId']
    #display(rated_movies)# user 189759 has 3 ratings
    unrated_movies = all_movies[~all_movies['movieId'].isin(rated_movies)]
    
    # sort user specific predictions in descending order, grab the top n
    top_n = unrated_movies.sort_values(str(user_id)+'_pred_rating', ascending=False).head(n)
    return top_n[['movieId', 'title', str(user_id)+'_pred_rating']]

recommend_unrated_top_n(189759, 5)